# Formal Methods Primer

## What This Is
Formal methods use mathematical logic to prove properties about programs and systems. For neural networks, we want to prove properties like:
- *Robustness*: For all inputs within ε-distance of x, the classification is the same
- *Safety*: The network output never exceeds a specified bound
- *Fairness*: Protected attributes don't influence the output

## Why Informal Testing Isn't Enough
Testing samples the input space. For an input space of 28×28 pixels (MNIST), there are 256^784 ≈ 10^1888 possible inputs. No amount of testing provides a guarantee — formal verification does.

## SAT/SMT Basics
**SAT (Satisfiability)**: Given a Boolean formula, does there exist an assignment of variables that makes it true?
**SMT (Satisfiability Modulo Theories)**: SAT extended with theories — linear arithmetic, arrays, bit vectors. Z3 is the leading SMT solver (Microsoft Research).

For neural networks, we encode network weights and activations as linear arithmetic constraints (feasible for piece-wise linear networks like ReLU).

In [ ]:
# Z3 basics: satisfiability and optimization
# pip install z3-solver

try:
    from z3 import *
    HAS_Z3 = True
except ImportError:
    HAS_Z3 = False
    print('z3-solver not installed. Showing Z3 concepts with pseudo-code.')

import numpy as np

if HAS_Z3:
    # Example 1: Basic satisfiability
    x, y = Ints('x y')
    solver = Solver()
    solver.add(x > 0)
    solver.add(y > 0)
    solver.add(x + y == 10)
    solver.add(x * 2 < y)
    
    result = solver.check()
    print(f'SAT result: {result}')
    if result == sat:
        m = solver.model()
        print(f'Solution: x={m[x]}, y={m[y]}')
        print(f'Verification: {m[x].as_long()} + {m[y].as_long()} = {m[x].as_long() + m[y].as_long()}')
    
    # Example 2: Proving a property (show UNSAT means property holds)
    solver2 = Solver()
    a, b = Reals('a b')
    # Try to find counterexample to: if a >= 0 and b >= 0, then a+b >= 0
    solver2.add(a >= 0, b >= 0)
    solver2.add(a + b < 0)  # negation of the property
    
    result2 = solver2.check()
    print(f'\nProperty verification: a>=0 ^ b>=0 => a+b>=0')
    print(f'Negation satisfiable? {result2} (unsat = property PROVED)')
else:
    print('Z3 Demo (conceptual):')
    print('  Solver: x > 0, y > 0, x + y = 10, 2x < y')
    print('  Solution: x=3, y=7 (SAT)')
    print('\n  Property: a>=0 ^ b>=0 => a+b>=0')
    print('  Negation: UNSAT (property PROVED)')


In [ ]:
# Encoding a small neural network as SMT constraints (conceptual)
# Demonstrates the key idea behind Reluplex/alpha-beta-CROWN

import numpy as np

class TinyReLUNet:
    """Tiny 2-layer ReLU network for verification demos."""
    
    def __init__(self):
        np.random.seed(42)
        # Layer 1: 2 inputs -> 4 hidden
        self.W1 = np.array([[0.5, 0.3], [-0.4, 0.8], [0.2, -0.6], [0.7, 0.1]])
        self.b1 = np.array([0.1, -0.1, 0.2, -0.2])
        # Layer 2: 4 hidden -> 2 outputs
        self.W2 = np.array([[0.4, -0.3, 0.6, 0.2], [-0.4, 0.3, -0.6, -0.2]])
        self.b2 = np.array([0.0, 0.0])
    
    def forward(self, x):
        h = np.maximum(0, self.W1 @ x + self.b1)
        return self.W2 @ h + self.b2
    
    def classify(self, x):
        return np.argmax(self.forward(x))

net = TinyReLUNet()

# Test classification
x0 = np.array([0.5, 0.3])
pred = net.classify(x0)
output = net.forward(x0)

print(f'Input: {x0}')
print(f'Network output: {output}')
print(f'Classification: class {pred}')

# Manual interval bound propagation (simplified CROWN)
# Given input box [x0 - eps, x0 + eps], compute output bounds
eps = 0.1
x_lo = x0 - eps
x_hi = x0 + eps

# Layer 1 pre-activations: W1 @ x + b1
# For linear layer: use interval arithmetic
W1p = np.maximum(0, net.W1)  # positive weights
W1n = np.minimum(0, net.W1)  # negative weights
pre1_lo = W1p @ x_lo + W1n @ x_hi + net.b1
pre1_hi = W1p @ x_hi + W1n @ x_lo + net.b1

# ReLU: output is [max(0, lo), max(0, hi)]
h_lo = np.maximum(0, pre1_lo)
h_hi = np.maximum(0, pre1_hi)

# Layer 2
W2p = np.maximum(0, net.W2)
W2n = np.minimum(0, net.W2)
out_lo = W2p @ h_lo + W2n @ h_hi + net.b2
out_hi = W2p @ h_hi + W2n @ h_lo + net.b2

print(f'\nInterval bound propagation (eps={eps}):')
print(f'Output lower bounds: {out_lo}')
print(f'Output upper bounds: {out_hi}')

# Check if classification is certified
# Class 0 is certified correct if out_lo[0] > out_hi[1]
certified = out_lo[pred] > max(out_hi[1-pred] for i in range(2) if i != pred)
print(f'\nRobustness certified for eps={eps}: {certified}')
print('(True = class {pred} is guaranteed for all inputs in the eps-ball)')
